# Occupation classification from job descriptions
**Or Shulrufer, Data Scientist assignment, Israel Central Bureau of Statistics**

This notebook implements a complete workflow on the supplied simulated data: loading, quality checks, model construction, evaluation and candidate suggestions for human review. The target is `SemelMishlachSofi`. The reported scores measure agreement with the supplied sample labels.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb)

**Codespaces:** open the notebook from the repository checkout and select Run all. The workbook is under `Datasets/cbs_occupation_survey_sample`.

**Colab or the emailed notebook:** upload `DUMMY_DATASET.xlsx` into the runtime Files panel, then select Runtime > Run all. Alternatively, use an existing repository checkout or set DATASETS_ROOT. In accordance with the repository structure, this notebook reads existing data. If the file is missing, it reports Dataset not found and skips dependent calculations. That outcome is not a successful model run. A CPU runtime is sufficient, with no API key required.

**GitHub code:** https://github.com/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb

**Execution evidence:** the Python cells were executed locally. The same computational code also completed a fresh Jupyter-kernel run in GitHub Actions, with 14 integrity checks and source checksum verification. [Recorded run](https://github.com/shulruferor-sudo/Advance-Data-Science/actions/runs/41051182364). The Colab link was opened and displayed the notebook. Execution within the Colab service itself remains unverified because the browser requires Google sign-in. Saved notebook outputs were produced by the local execution.


## 1. Problem definition and decisions before evaluation
The business objective is to help coders choose an occupation code from a job description, reduce repeated work and route incomplete or ambiguous cases for review. Future operational evaluation should measure coding quality and handling time.

The primary model was fixed before evaluation: word and character TF-IDF with a LinearSVC classifier using C=1. This representation captures patterns in the text. It does not guarantee semantic understanding. Comparisons include a most-frequent-class baseline, word-only features, character-only features and core-description-only features. Parameters are not changed in response to evaluation scores.

Primary inputs include the job description, activities, duties and organizational context. Record identifiers, occupation codes and industry classification codes are excluded. Age, locality, education and job start dates are also excluded from this demonstration. Availability of the industry code at prediction time was unspecified. Metadata defines ID as a serial number rather than a person identifier. The job start date is distinct from survey submission time.

Identical core descriptions remain in the same evaluation group, using EzoAvoda, TeurPeula and TeurTafkid after basic normalization. This controls exact text repetition. It does not establish respondent identity, and similar wording may remain in different groups.

## Runtime setup
The following cell checks dependencies, configures paths and reports versions.


In [1]:
import sys, subprocess, importlib.util, json, re, unicodedata, hashlib, time
from pathlib import Path
import os
from contextlib import contextmanager
from datetime import datetime, timezone

# Install only missing dependencies. Colab usually already provides these.
for module, package in [("pandas", "pandas==2.2.3"), ("numpy", "numpy==2.3.5"),
                        ("sklearn", "scikit-learn==1.8.0"), ("openpyxl", "openpyxl==3.1.5")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
import numpy as np
import pandas as pd
import sklearn, openpyxl
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
REPOSITORY_ROOT = Path(os.environ.get("ADVANCED_DATA_SCIENCE_ROOT", Path.cwd())).expanduser().resolve()
for candidate in (REPOSITORY_ROOT, *REPOSITORY_ROOT.parents):
    if (candidate / "Datasets").is_dir() and (candidate / "project_notebooks").is_dir():
        REPOSITORY_ROOT = candidate
        break
DATASETS_ROOT = Path(os.environ.get("DATASETS_ROOT", REPOSITORY_ROOT / "Datasets")).expanduser()
PROJECT_DIRECTORY = REPOSITORY_ROOT / "project_notebooks" / "cbs_occupation_classification"
if not PROJECT_DIRECTORY.is_dir():
    PROJECT_DIRECTORY = Path.cwd()
OUTPUT = PROJECT_DIRECTORY / "results"
OUTPUT.mkdir(exist_ok=True)
VERSIONS = {"python": sys.version.split()[0], "pandas": pd.__version__,
            "numpy": np.__version__, "scikit-learn": sklearn.__version__,
            "openpyxl": openpyxl.__version__}
print("Environment:", json.dumps(VERSIONS))
DECLARED_VERSIONS={"pandas":"2.2.3","numpy":"2.3.5","scikit-learn":"1.8.0","openpyxl":"3.1.5"}
for package,version in DECLARED_VERSIONS.items():
    if VERSIONS[package]!=version:
        print(f"Version difference: {package}={VERSIONS[package]}, reference={version}. Existing installation retained.")

@contextmanager
def report_failure(stage):
    try:
        yield
    except Exception as error:
        print(f"FAILED: {stage}: {type(error).__name__}: {error}")
        print("Inspect the stated input, dependency or operation before rerunning dependent cells.")
        raise


Environment: {"python": "3.12.14", "pandas": "2.2.3", "numpy": "2.3.5", "scikit-learn": "1.8.0", "openpyxl": "3.1.5"}


## 2. Source loading and structural checks
Read the 18 defined fields in Survey Dataset and the four definition columns in Metadata. The workbook also contains a formula in J202 that is not a survey record, and copies of a field name in distant Metadata columns. Preserve the original workbook. Rows without a valid ID are recorded separately and excluded from training.


In [ ]:
from urllib.request import urlretrieve
from IPython.display import display

DATA_URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
urlretrieve(DATA_URL, "DUMMY_DATASET.xlsx")

sheets = pd.read_excel("DUMMY_DATASET.xlsx", sheet_name=None)
df = sheets["Survey Dataset"]
metadata = sheets["Metadata"]

print("Survey Dataset:")
display(df.head(20))

print("Metadata:")
display(metadata.head(20))


In [2]:
with report_failure("Read supplied survey"):
    DATA_PATH = DATASETS_ROOT / "cbs_occupation_survey_sample" / "DUMMY_DATASET.xlsx"
    if not DATA_PATH.is_file() and Path("DUMMY_DATASET.xlsx").is_file():
        DATA_PATH = Path("DUMMY_DATASET.xlsx")

    EXPECTED_COLUMNS = ["ID", "YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka",
        "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid", "MenahelEtMi", "MaamadAvoda",
        "MakorSachar", "TeudaGvoha", "shnotlimud", "Gil", "TarichHathala", "SemelAnafSofi", "SemelMishlachSofi"]

    def read_survey(path):
        book = pd.ExcelFile(path, engine="openpyxl")
        required = {"Survey Dataset", "Metadata"}
        if not required.issubset(book.sheet_names):
            raise ValueError("Missing required worksheet: " + str(required - set(book.sheet_names)))
        raw = pd.read_excel(book, sheet_name="Survey Dataset", usecols="A:R", dtype=str).fillna("")
        if list(raw.columns) != EXPECTED_COLUMNS:
            raise ValueError("Input schema differs from the supplied assignment workbook.")
        valid = raw["ID"].str.fullmatch(r"[1-9]\d*")
        excluded = raw.loc[~valid].copy()
        data = raw.loc[valid].copy().reset_index(drop=True)
        if data.empty or data["ID"].duplicated().any():
            raise ValueError("No usable records or duplicate IDs. Inspect the input before training.")
        metadata = pd.read_excel(book, sheet_name="Metadata", usecols="A:D", dtype=str).fillna("")
        return data, excluded, metadata

    df = None
    try:
        df, excluded, metadata = read_survey(DATA_PATH)
    except FileNotFoundError:
        print(f"Dataset not found: {DATA_PATH}. Upload the supplied workbook or prepare it in Datasets. Dependent calculations are skipped.")
    if df is not None:
        DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
        print("Dataset SHA-256:", DATA_SHA256)
        print("Survey records:", len(df), "Columns:", len(df.columns), "Excluded nonrecords:", len(excluded))
        print(metadata.to_string(index=False))
        print("Excluded populated cells:", [{k:v for k,v in row.items() if v} for row in excluded.to_dict("records")])


Dataset SHA-256: e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac
Survey records: 200 Columns: 18 Excluded nonrecords: 1
            משתנה                                                                                                  הסבר   פורמט                                                                                                                                                                                                                                                                                                                             ערכים אפשריים
               ID                                                                                               מספר רץ INTEGER                                                                                                                                                                                                                                                                                      

## 3. Normalization, partial codes and repeated descriptions
Normalization is limited to Unicode NFKC, lowercasing and whitespace consolidation. Words, facts and labels remain unchanged. The original abbreviation meaning "ditto" is retained because its reference is not always clear.

A complete code contains four digits. A code containing X is retained for candidate assistance and review. It does not provide a complete target for supervised training or exact-code evaluation. The supplied labels have not been independently adjudicated, so agreement with them does not establish the true coding error rate.


In [3]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 3"):
        TARGET = "SemelMishlachSofi"
        TEXT_FIELDS = ["ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
        CORE_FIELDS = ["EzoAvoda", "TeurPeula", "TeurTafkid"]

        def normalize(value):
            if pd.isna(value):
                return ""
            return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value)).strip().lower())

        def text_for(frame, fields=TEXT_FIELDS):
            # Separate field boundaries. Do not infer the meaning of missing values or 'ditto'.
            return frame[fields].fillna("").apply(lambda row: " \n ".join(normalize(v) for v in row), axis=1)

        def group_for(frame):
            signatures = frame[CORE_FIELDS].apply(lambda row: tuple(normalize(v) for v in row), axis=1)
            # If all task fields are empty, use available organizational text conservatively.
            for i, signature in signatures.items():
                if not any(signature):
                    signatures.at[i] = tuple(normalize(frame.at[i, c]) for c in TEXT_FIELDS)
            return signatures.map(lambda x: hashlib.sha256(json.dumps(x, ensure_ascii=False).encode()).hexdigest())

        labels = df[TARGET].str.strip().str.upper()
        complete_mask = labels.str.fullmatch(r"\d{4}")
        partial_mask = labels.str.fullmatch(r"[0-9X]{4}") & labels.str.contains("X")
        if not (complete_mask | partial_mask).all():
            raise ValueError("Unexpected label format. Inspect and document rather than silently repairing it.")
        complete = df.loc[complete_mask].reset_index(drop=True)
        partial = df.loc[partial_mask].reset_index(drop=True)
        y = complete[TARGET].str.strip().to_numpy()
        groups = group_for(complete).to_numpy()
        classes = np.sort(np.unique(y))
        counts = pd.Series(y).value_counts()
        feature_cols = [c for c in EXPECTED_COLUMNS if c not in {"ID", TARGET}]
        duplicate_sets = []
        for _, block in df.groupby(feature_cols, dropna=False, sort=False):
            if len(block) > 1 and block[TARGET].nunique() > 1:
                duplicate_sets.append({"ids": block.ID.tolist(), "labels": block[TARGET].tolist()})
        quality = {
            "records":len(df), "fields":len(df.columns), "complete_labels":len(complete),
            "partial_labels":len(partial), "complete_classes":len(classes),
            "singleton_classes":int((counts == 1).sum()), "task_groups":int(len(np.unique(groups))),
            "excluded_nonrecords":len(excluded),
            "duplicate_feature_rows":int(df[feature_cols].duplicated(keep=False).sum()),
            "conflicting_exact_feature_sets":duplicate_sets,
            "missing_by_field":{c:int(df[c].eq("").sum()) for c in EXPECTED_COLUMNS},
            "partial_codes":partial[TARGET].value_counts().to_dict()
        }
        print(json.dumps(quality, ensure_ascii=False, indent=2))
        print("Complete-label class counts:")
        print(counts.to_string())


{
  "records": 200,
  "fields": 18,
  "complete_labels": 191,
  "partial_labels": 9,
  "complete_classes": 59,
  "singleton_classes": 24,
  "task_groups": 90,
  "excluded_nonrecords": 1,
  "duplicate_feature_rows": 22,
  "conflicting_exact_feature_sets": [
    {
      "ids": [
        "2",
        "33",
        "65",
        "96"
      ],
      "labels": [
        "3311",
        "3211",
        "3211",
        "3211"
      ]
    }
  ],
  "missing_by_field": {
    "ID": 0,
    "YeshuvAvoda": 0,
    "ShemAvoda": 0,
    "SugAvoda": 25,
    "ShemMachlaka": 64,
    "SugMachlaka": 134,
    "EzoAvoda": 7,
    "TeurPeula": 136,
    "TeurTafkid": 172,
    "MenahelEtMi": 168,
    "MaamadAvoda": 2,
    "MakorSachar": 14,
    "TeudaGvoha": 2,
    "shnotlimud": 2,
    "Gil": 2,
    "TarichHathala": 0,
    "SemelAnafSofi": 0,
    "SemelMishlachSofi": 0
  },
  "partial_codes": {
    "1XXX": 4,
    "XXXX": 4,
    "335X": 1
  }
}
Complete-label class counts:
5223    14
9112    12
2512    10
5120     7

## 4. Evaluation with description-group separation
Each of the 191 completely coded records receives a prediction from a model trained without its group. Five-fold GroupKFold keeps groups separate. In every fold, the vocabulary and TF-IDF weights are also learned from training data alone.

**Metrics:** accuracy is the proportion of exact full-code matches. Macro F1 weights each of the 59 classes equally and exposes difficulty with rare classes. Top-3 accuracy is the proportion whose supplied label occurs among the three candidates. First-digit agreement is also reported and measures a coarser match than complete coding.

Some evaluation folds contain codes absent from their training data. These cases expose the limits of a classifier restricted to learned classes and remain in the primary calculation. Accuracy conditional on the class having appeared in training is reported separately. Groups represent text descriptions rather than people. The evaluation concerns this supplied sample rather than a prospective production population.


In [4]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 4"):
        def make_model(kind="combined"):
            word = TfidfVectorizer(ngram_range=(1,2), token_pattern=r"(?u)\b\w+\b", sublinear_tf=True)
            char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), sublinear_tf=True)
            if kind == "dummy":
                return DummyClassifier(strategy="most_frequent")
            feature = word if kind == "word" else char if kind == "char" else FeatureUnion([("word",word),("char",char)])
            return Pipeline([("features",feature), ("classifier",LinearSVC(C=1.0, dual=True, random_state=SEED, max_iter=20000))])

        def ranks_and_margin(model, text):
            scores = np.asarray(model.decision_function(text))
            if scores.ndim == 1:
                scores = np.column_stack([-scores, scores])
            rank = np.argsort(-scores, axis=1, kind="stable")[:, :min(3, len(model.classes_))]
            ranked = model.classes_[rank]
            ordered = np.sort(scores, axis=1)
            margins = ordered[:,-1] - ordered[:,-2]
            return ranked, margins

        def evaluate(kind, fields=TEXT_FIELDS, split_kind="grouped"):
            text = text_for(complete, fields).to_numpy()
            if split_kind == "grouped":
                splits = GroupKFold(n_splits=5).split(text,y,groups)
            else:
                splits = KFold(n_splits=5,shuffle=True,random_state=SEED).split(text)
            rows, fold_stats = [], []
            for fold,(train_idx,test_idx) in enumerate(splits,1):
                shared_groups = set(groups[train_idx]) & set(groups[test_idx])
                if split_kind == "grouped":
                    assert not shared_groups
                model=make_model(kind)
                started=time.perf_counter()
                if kind == "dummy":
                    model.fit(np.zeros((len(train_idx),1)),y[train_idx])
                    pred=model.predict(np.zeros((len(test_idx),1)))
                    ranked=pred[:,None]
                    margins=np.full(len(test_idx),np.nan)
                else:
                    model.fit(text[train_idx],y[train_idx])
                    pred=model.predict(text[test_idx])
                    ranked,margins=ranks_and_margin(model,text[test_idx])
                known=np.isin(y[test_idx],model.classes_)
                fold_stats.append({"fold":fold,"train_n":len(train_idx),"test_n":len(test_idx),
                    "test_groups_seen_in_training":len(shared_groups),"unseen_label_rows":int((~known).sum()),
                    "accuracy":float(accuracy_score(y[test_idx],pred)),"runtime_seconds":time.perf_counter()-started})
                for j,idx in enumerate(test_idx):
                    rows.append({"ID":complete.iloc[idx].ID,"fold":fold,"group":groups[idx],
                        "observed_code":y[idx],"predicted_code":pred[j],"top_2":ranked[j,1] if ranked.shape[1]>1 else "",
                        "top_3":ranked[j,2] if ranked.shape[1]>2 else "","top3_hit":bool(y[idx] in ranked[j]),
                        "label_seen_in_training":bool(known[j]),"correct":bool(y[idx]==pred[j]),
                        "margin_not_probability":float(margins[j])})
            result=pd.DataFrame(rows).sort_values("ID",key=lambda s:s.astype(int)).reset_index(drop=True)
            assert len(result)==len(complete) and result.ID.nunique()==len(complete)
            truth=result.observed_code.to_numpy()
            prediction=result.predicted_code.to_numpy()
            known=result.label_seen_in_training
            metrics={"model":kind,"fields":"core" if fields==CORE_FIELDS else "context_and_core", "split":split_kind,
                "n":len(result),"accuracy":float(accuracy_score(truth,prediction)),
                "macro_f1_all_59_classes":float(f1_score(truth,prediction,labels=classes,average="macro",zero_division=0)),
                "top3_accuracy":float(result.top3_hit.mean()) if kind!="dummy" else None,
                "first_digit_accuracy":float(np.mean([a[0]==b[0] for a,b in zip(truth,prediction)])),
                "unseen_label_rows":int((~known).sum()),
                "accuracy_seen_labels_only":float(result.loc[known,"correct"].mean()) if known.any() else None}
            return result,metrics,fold_stats

        experiments=[("dummy",TEXT_FIELDS,"grouped"), ("word",TEXT_FIELDS,"grouped"),
            ("char",TEXT_FIELDS,"grouped"), ("combined",TEXT_FIELDS,"grouped"),
            ("combined",CORE_FIELDS,"grouped"), ("combined",TEXT_FIELDS,"random_rows")]
        all_metrics=[]
        experiment_folds={}
        for kind,fields,split_kind in experiments:
            predictions,metrics,folds=evaluate(kind,fields,split_kind)
            all_metrics.append(metrics)
            key=kind+"_"+metrics["fields"]+"_"+split_kind
            experiment_folds[key]=folds
            if key=="combined_context_and_core_grouped":
                primary_predictions=predictions.copy()
                primary_metrics=metrics.copy()
        metrics_table=pd.DataFrame(all_metrics)
        print(metrics_table.to_string(index=False,float_format=lambda x:f"{x:.4f}"))
        print("Primary fold diagnostics:")
        print(pd.DataFrame(experiment_folds["combined_context_and_core_grouped"]).to_string(index=False))
        print("Random-row diagnostics, shown only to reveal overlap:")
        print(pd.DataFrame(experiment_folds["combined_context_and_core_random_rows"])[["fold","test_groups_seen_in_training","unseen_label_rows"]].to_string(index=False))


   model           fields       split   n  accuracy  macro_f1_all_59_classes  top3_accuracy  first_digit_accuracy  unseen_label_rows  accuracy_seen_labels_only
   dummy context_and_core     grouped 191    0.0419                   0.0026            NaN                0.3141                 84                     0.0748
    word context_and_core     grouped 191    0.3717                   0.2110         0.4084                0.5654                 84                     0.6636
    char context_and_core     grouped 191    0.3979                   0.2228         0.4869                0.5393                 84                     0.7103
combined context_and_core     grouped 191    0.3979                   0.2232         0.4712                0.5707                 84                     0.7103
combined             core     grouped 191    0.3717                   0.1899         0.4398                0.5288                 84                     0.6636
combined context_and_core random_rows 19

## 5. Error interpretation and limitations
Evaluation scores measure agreement with the supplied labels, some of which may be incorrect. Disagreement alone does not justify changing a label. Review a disputed record against classification rules and a coder's assessment.

A random row split allows some job descriptions to recur in training and evaluation. Comparing it with group separation is a sensitivity analysis rather than an independent estimate of production performance. Separation by identical descriptions also leaves possible similarity between groups.

All class scores and errors are displayed below. The primary model remains the model selected before evaluation, even if another variant scores higher on a metric.


In [5]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 5"):
        report=classification_report(primary_predictions.observed_code,primary_predictions.predicted_code,
            labels=classes,output_dict=True,zero_division=0)
        print(pd.DataFrame(report).transpose().to_string(float_format=lambda x:f"{x:.3f}"))
        errors=primary_predictions.loc[~primary_predictions.correct].copy()
        print("Errors:",len(errors),"of",len(primary_predictions))
        print(errors[["ID","observed_code","predicted_code","label_seen_in_training"]].to_string(index=False))
        # Equal weight per held-out task group is a sensitivity metric for repeated templates.
        group_accuracy=primary_predictions.groupby("group").correct.mean()
        print("Equal-group average accuracy:",round(float(group_accuracy.mean()),4))
        # Cluster bootstrap is descriptive split sensitivity, not a population confidence guarantee.
        rng=np.random.default_rng(SEED)
        boot=np.array([rng.choice(group_accuracy.to_numpy(),size=len(group_accuracy),replace=True).mean() for _ in range(2000)])
        print("Descriptive group bootstrap central 95% interval:",np.quantile(boot,[.025,.975]).round(4))


              precision  recall  f1-score  support
1212              0.000   0.000     0.000    1.000
1221              0.000   0.000     0.000    1.000
1330              0.000   0.000     0.000    1.000
1345              0.000   0.000     0.000    1.000
2113              0.000   0.000     0.000    1.000
2120              0.750   1.000     0.857    3.000
2131              0.750   1.000     0.857    3.000
2142              1.000   0.500     0.667    2.000
2262              0.000   0.000     0.000    2.000
2330              0.000   0.000     0.000    1.000
2342              0.714   1.000     0.833    5.000
2411              0.000   0.000     0.000    1.000
2412              0.000   0.000     0.000    1.000
2423              0.000   0.000     0.000    1.000
2431              0.000   0.000     0.000    1.000
2511              0.333   0.143     0.200    7.000
2512              0.556   1.000     0.714   10.000
2514              0.000   0.000     0.000    4.000
2631              1.000   0.333

## 6. Suggestions for partially coded records
Fit the primary model on the 191 completely coded records. This demonstration model produces suggestions for the nine partial records. A pattern such as 335X permits only candidates preserving the known digits. If no learned class is compatible, leave the suggestion empty. Every candidate requires human review. These records lack complete reference labels, so exact-code accuracy cannot be measured for them.

A LinearSVC decision score is not a probability. A large gap between candidate scores does not authorize automatic acceptance. When fields are missing, the model may rely primarily on organizational context.


In [6]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 6"):
        final_model=make_model("combined")
        final_model.fit(text_for(complete).to_numpy(),y)

        def compatible(code,pattern):
            return len(code)==len(pattern)==4 and all(p=="X" or p==c for c,p in zip(code,pattern))

        def suggest(frame,patterns=None):
            frame=frame.copy()
            for col in TEXT_FIELDS:
                if col not in frame:
                    frame[col]=""
            text=text_for(frame).to_numpy()
            scores=np.asarray(final_model.decision_function(text))
            if scores.ndim==1:
                scores=np.column_stack([-scores,scores])
            output=[]
            for pos,(_,row) in enumerate(frame.iterrows()):
                pattern=normalize(patterns[pos]).upper() if patterns is not None else "XXXX"
                if not re.fullmatch(r"[0-9X]{4}",pattern):
                    raise ValueError("Expected a four-character code pattern with digits or X.")
                allowed=[i for i,c in enumerate(final_model.classes_) if compatible(c,pattern)]
                ranked=sorted(allowed,key=lambda j:(-scores[pos,j],str(final_model.classes_[j])))[:3]
                candidates=[str(final_model.classes_[j]) for j in ranked]
                has_text=any(normalize(row[c]) for c in TEXT_FIELDS)
                if not has_text:
                    candidates=[]
                no_core=not any(normalize(row.get(c,"")) for c in CORE_FIELDS)
                reason="missing_all_text" if not has_text else "no_compatible_training_class" if not candidates else "missing_task_description" if no_core else "incomplete_reference_label"
                output.append({"ID":str(row.get("ID","")),"original_pattern":pattern,
                    "suggestion_1":candidates[0] if candidates else "",
                    "suggestion_2":candidates[1] if len(candidates)>1 else "",
                    "suggestion_3":candidates[2] if len(candidates)>2 else "",
                    "review_required":True,"review_reason":reason})
            return pd.DataFrame(output)

        partial_suggestions=suggest(partial,partial[TARGET].tolist())
        print(partial_suggestions.to_string(index=False))
        print("All suggestions require human review. No labels were changed.")


 ID original_pattern suggestion_1 suggestion_2 suggestion_3  review_required              review_reason
 18             1XXX         1345         1212         1221             True incomplete_reference_label
 45             XXXX         5131         5120         9412             True   missing_task_description
 50             1XXX         1345         1212         1221             True incomplete_reference_label
 81             1XXX         1345         1212         1221             True incomplete_reference_label
108             XXXX         5131         5120         9412             True   missing_task_description
113             1XXX         1345         1212         1221             True incomplete_reference_label
142             XXXX         5230         8183         5153             True   missing_task_description
165             335X         3354                                       True incomplete_reference_label
193             XXXX         3354         3512         7126     

## 7. Integrity checks and outputs
The checks cover empty inputs, unseen words, preservation of known digits in partial codes, leading zeros, absence of compatible classes and group separation. They establish the tested implementation behavior. Real-population validity and correctness of the original coding require separate evidence.


In [7]:
if df is None:
    print("Skipped: dataset not available.")
else:
    with report_failure("Assignment stage 7"):
        checks={}
        checks["original_200_records"] = len(df)==200
        checks["label_partition"] = len(complete)==191 and len(partial)==9
        checks["complete_classes_59"] = len(classes)==59
        checks["exactly_one_heldout_prediction_per_record"] = len(primary_predictions)==191 and primary_predictions.ID.nunique()==191
        checks["no_primary_group_overlap"] = all(f["test_groups_seen_in_training"]==0 for f in experiment_folds["combined_context_and_core_grouped"])
        checks["no_target_or_postcoding_features"] = TARGET not in TEXT_FIELDS and "SemelAnafSofi" not in TEXT_FIELDS and "ID" not in TEXT_FIELDS
        checks["leading_zero_preserved"] = normalize("0110")=="0110" and compatible("0110","0XXX")
        checks["partial_pattern_constraints"] = compatible("3354","335X") and not compatible("3311","335X")
        blank=suggest(pd.DataFrame([{c:"" for c in TEXT_FIELDS}]))
        checks["empty_text_abstains"] = blank.suggestion_1.eq("").all() and blank.review_required.all()
        unknown=suggest(pd.DataFrame([{"EzoAvoda":"zzqxv newtoken\u05dc\u05d0\u05de\u05d5\u05db\u05e8"}]))
        checks["unknown_words_run_and_require_review"] = len(unknown)==1 and unknown.review_required.all()
        none_allowed=suggest(pd.DataFrame([{"EzoAvoda":"\u05de\u05ea\u05db\u05e0\u05ea"}]),["0000"])
        checks["no_compatible_class_abstains"] = none_allowed.suggestion_1.eq("").all()
        checks["prefixes_preserved_in_outputs"] = all(compatible(row[c],row.original_pattern) for _,row in partial_suggestions.iterrows() for c in ["suggestion_1","suggestion_2","suggestion_3"] if row[c])
        checks["all_partial_rows_require_review"] = bool(partial_suggestions.review_required.all())
        try:
            suggest(pd.DataFrame([{"EzoAvoda":"\u05de\u05ea\u05db\u05e0\u05ea"}]),["invalid"])
            checks["invalid_pattern_rejected"] = False
        except ValueError:
            checks["invalid_pattern_rejected"] = True
        checks={k:bool(v) for k,v in checks.items()}
        if not all(checks.values()):
            raise AssertionError({k:v for k,v in checks.items() if not v})
        print(json.dumps(checks,indent=2))

        primary_predictions.to_csv(OUTPUT/"heldout_predictions.csv",index=False,encoding="utf-8-sig")
        partial_suggestions.to_csv(OUTPUT/"partial_label_suggestions.csv",index=False,encoding="utf-8-sig")
        metrics_table.to_csv(OUTPUT/"model_comparison.csv",index=False,encoding="utf-8-sig")
        run={"dataset_sha256":DATA_SHA256,"versions":VERSIONS,"seed":SEED,"quality":quality,
             "primary_metrics":primary_metrics,"experiments":all_metrics,"folds":experiment_folds,
             "equal_group_accuracy":float(group_accuracy.mean()),
             "descriptive_group_bootstrap_interval":np.quantile(boot,[.025,.975]).tolist(),
             "checks":checks,"github_url":"https://github.com/shulruferor-sudo/Advance-Data-Science/blob/main/project_notebooks/cbs_occupation_classification/Assignment.ipynb","colab_execution_verified":False,
             "model_choice":"Combined text SVC fixed before evaluation. No tuning on held-out predictions.",
             "limitations":["Simulated sample", "No respondent identifier or submission timestamp", "No adjudicated gold labels",
                            "Near duplicates may span groups", "Closed class set", "No independent final test set", "No Colab runtime execution yet"]}
        (OUTPUT/"run_results.json").write_text(json.dumps(run,ensure_ascii=False,indent=2),encoding="utf-8")
        print("Results saved in",OUTPUT.resolve())
        print("Source checksum unchanged:",hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()==DATA_SHA256)


{
  "original_200_records": true,
  "label_partition": true,
  "complete_classes_59": true,
  "exactly_one_heldout_prediction_per_record": true,
  "no_primary_group_overlap": true,
  "no_target_or_postcoding_features": true,
  "leading_zero_preserved": true,
  "partial_pattern_constraints": true,
  "empty_text_abstains": true,
  "unknown_words_run_and_require_review": true,
  "no_compatible_class_abstains": true,
  "prefixes_preserved_in_outputs": true,
  "all_partial_rows_require_review": true,
  "invalid_pattern_rejected": true
}
Results saved in /workspace/scratch/98c53c9539cb/assignment-repo/project_notebooks/cbs_occupation_classification/results
Source checksum unchanged: True


### Additional classical alternatives

These exploratory comparisons were added after the original evaluation. They keep the same 191 records, five task-group folds and complete 59-class scoring set. The preselected combined TF-IDF/SVC remains the submission demonstration model, not a claim of optimality.


In [ ]:
# Author: /root. Scientific method integration. UTC 2026-10-30.
# These additional comparisons were requested after the original six experiments.
# They are exploratory and do not change the preselected submission model.
if df is None:
    print("Skipped additional comparisons: dataset not available.")
else:
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.exceptions import ConvergenceWarning
    import warnings

    alternative_rows = []
    alternative_predictions = []
    fixed_splits = list(GroupKFold(n_splits=5).split(text_for(complete), y, groups))
    texts = text_for(complete).to_numpy()
    for model_name in ["logistic_regression", "multinomial_nb", "cosine_nearest_record"]:
        fold_rows = []
        for fold, (train_idx, test_idx) in enumerate(fixed_splits, 1):
            assert not set(groups[train_idx]) & set(groups[test_idx])
            features = make_model("combined").named_steps["features"]
            x_train = features.fit_transform(texts[train_idx])
            x_test = features.transform(texts[test_idx])
            if model_name == "cosine_nearest_record":
                similarity = cosine_similarity(x_test, x_train)
                # Deterministic tie break: first training row in original order.
                prediction = y[train_idx[np.argmax(similarity, axis=1)]]
            else:
                classifier = (LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000, random_state=SEED)
                              if model_name == "logistic_regression" else MultinomialNB(alpha=1.0))
                with warnings.catch_warnings():
                    warnings.simplefilter("error", ConvergenceWarning)
                    classifier.fit(x_train, y[train_idx])
                prediction = classifier.predict(x_test)
            for pos, idx in enumerate(test_idx):
                fold_rows.append({"model": model_name, "ID": str(complete.iloc[idx].ID), "fold": fold,
                                  "observed_code": str(y[idx]), "predicted_code": str(prediction[pos]),
                                  "label_seen_in_training": bool(y[idx] in y[train_idx]),
                                  "correct": bool(y[idx] == prediction[pos])})
        frame = pd.DataFrame(fold_rows)
        assert len(frame) == 191 and frame.ID.nunique() == 191
        assert int((~frame.label_seen_in_training).sum()) == 84
        alternative_predictions.append(frame)
        alternative_rows.append({"model": model_name, "correct": int(frame.correct.sum()), "n": len(frame),
            "accuracy": float(frame.correct.mean()),
            "macro_f1_all_59_classes": float(f1_score(frame.observed_code, frame.predicted_code,
                labels=classes, average="macro", zero_division=0)), "unseen_label_rows": 84})
    alternative_comparison = pd.DataFrame(alternative_rows)
    alternative_comparison.to_csv(OUTPUT / "alternative_model_comparison.csv", index=False)
    pd.concat(alternative_predictions, ignore_index=True).to_csv(OUTPUT / "alternative_predictions.csv", index=False)
    alternative_checks = {
        "three_additional_models": len(alternative_comparison) == 3,
        "same_primary_record_fold_mapping": all(
            frame.set_index("ID").fold.to_dict() == primary_predictions.set_index("ID").fold.to_dict()
            for frame in alternative_predictions),
        "all_records_retained": all(len(frame) == 191 and frame.ID.nunique() == 191 for frame in alternative_predictions),
        "source_checksum_unchanged": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == DATA_SHA256}
    assert all(alternative_checks.values()), alternative_checks
    additional_run = {"purpose": "Exploratory comparisons added after the original evaluation",
        "selection_rule": "Preselected combined SVC unchanged. No hyperparameter search.",
        "parameters": {"logistic_regression": {"C": 1.0, "solver": "lbfgs", "max_iter": 2000},
            "multinomial_nb": {"alpha": 1.0}, "cosine_nearest_record": {"neighbors": 1, "tie_break": "first training row"}},
        "representation": "Same train-only word and character TF-IDF as the combined SVC",
        "models": alternative_rows, "checks": alternative_checks, "dataset_sha256": DATA_SHA256}
    (OUTPUT / "alternative_run_results.json").write_text(json.dumps(additional_run, indent=2), encoding="utf-8")
    print(alternative_comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("Additional integrity checks:", alternative_checks)


### Modern multilingual encoder comparison

These exploratory comparisons were added after the original evaluation. They keep the same 191 records, five task-group folds and complete 59-class scoring set. The preselected combined TF-IDF/SVC remains the submission demonstration model, not a claim of optimality.

The frozen Qwen3-Embedding-0.6B checkpoint is pinned to a specific revision. Hebrew support is documented by its authors. A linear head is fitted separately in each fold. The first run downloads public model weights and may take several minutes on CPU. Set the environment variable RUN_MODERN_ENCODER=0 before Run all to run only the classical comparisons. No survey text is sent to an inference service.

Sources: https://github.com/QwenLM/Qwen3-Embedding and https://huggingface.co/Qwen/Qwen3-Embedding-0.6B . Publisher leaderboard scores date from 2025 and are not assignment results.


In [ ]:
# Author: /root. Modern encoder comparison. UTC 2026-10-30.
# First execution downloads public model weights. Survey data stay in this runtime.
RUN_MODERN_ENCODER = os.environ.get("RUN_MODERN_ENCODER", "1") == "1"
if df is None or not RUN_MODERN_ENCODER:
    print("Modern encoder comparison skipped. Set RUN_MODERN_ENCODER=1 to run it.")
else:
    if importlib.util.find_spec("torch") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cpu"])
    if importlib.util.find_spec("transformers") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1"])
    import torch, transformers
    from transformers import AutoModel, AutoTokenizer
    from sklearn.metrics.pairwise import cosine_similarity
    MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
    MODEL_REVISION = "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"
    INSTRUCTION = "Represent this Hebrew job description for classifying its occupation from its work tasks and duties."
    torch.manual_seed(SEED)
    torch.set_num_threads(min(4, os.cpu_count() or 1))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, padding_side="left", trust_remote_code=False)
    encoder = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False,
                                        torch_dtype=torch.float32).to(device).eval()
    inputs = [f"Instruct: {INSTRUCTION}\nQuery: {text}" for text in text_for(complete).to_list()]
    token_lengths = [len(tokenizer.encode(text)) for text in inputs]
    # Stop instead of silently truncating job descriptions.
    assert max(token_lengths) <= 1024, "Input exceeds the declared comparison token limit."
    embeddings = []
    started = time.perf_counter()
    with torch.inference_mode():
        for offset in range(0, len(inputs), 8):
            tokens = tokenizer(inputs[offset:offset+8], padding=True, truncation=False, return_tensors="pt").to(device)
            output = encoder(**tokens, use_cache=False).last_hidden_state[:, -1, :]
            embeddings.append(torch.nn.functional.normalize(output, p=2, dim=1).cpu().numpy())
    embeddings = np.concatenate(embeddings)
    assert embeddings.shape == (191, 1024) and np.isfinite(embeddings).all()
    # The encoder is frozen: no sample labels or corpus fitting affect embeddings.
    # Only the supervised head is trained within each training fold.
    modern_rows = []
    for fold, (train_idx, test_idx) in enumerate(GroupKFold(n_splits=5).split(embeddings, y, groups), 1):
        assert not set(groups[train_idx]) & set(groups[test_idx])
        head = LinearSVC(C=1.0, dual=True, random_state=SEED, max_iter=20000)
        head.fit(embeddings[train_idx], y[train_idx])
        predictions = head.predict(embeddings[test_idx])
        for pos, idx in enumerate(test_idx):
            modern_rows.append({"ID": str(complete.iloc[idx].ID), "fold": fold, "observed_code": str(y[idx]),
                "predicted_code": str(predictions[pos]), "correct": bool(y[idx] == predictions[pos]),
                "label_seen_in_training": bool(y[idx] in head.classes_)})
    modern_predictions = pd.DataFrame(modern_rows)
    modern_checks = {
        "one_prediction_per_record": len(modern_predictions) == 191 and modern_predictions.ID.nunique() == 191,
        "same_fold_assignment": modern_predictions.set_index("ID").fold.to_dict() == primary_predictions.set_index("ID").fold.to_dict(),
        "same_unseen_label_count": int((~modern_predictions.label_seen_in_training).sum()) == 84,
        "no_input_truncation": max(token_lengths) <= 1024,
        "normalized_finite_embeddings": bool(np.isfinite(embeddings).all() and np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)),
        "source_unchanged": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == DATA_SHA256}
    assert all(modern_checks.values()), modern_checks
    modern_result = {"model": MODEL_ID, "revision": MODEL_REVISION, "instruction": INSTRUCTION,
        "encoder_frozen": True, "head": "LinearSVC C=1.0", "device": device, "dtype": "float32",
        "correct": int(modern_predictions.correct.sum()), "n": 191, "accuracy": float(modern_predictions.correct.mean()),
        "macro_f1_all_59_classes": float(f1_score(modern_predictions.observed_code, modern_predictions.predicted_code,
            labels=classes, average="macro", zero_division=0)), "unseen_label_rows": 84,
        "max_input_tokens": max(token_lengths), "runtime_seconds": time.perf_counter()-started,
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__}, "checks": modern_checks,
        "scope": "Exploratory frozen representation comparison. Not fine-tuning or a benchmark of all modern models."}
    modern_predictions.to_csv(OUTPUT / "modern_encoder_predictions.csv", index=False)
    (OUTPUT / "modern_encoder_results.json").write_text(json.dumps(modern_result, indent=2), encoding="utf-8")
    print(json.dumps(modern_result, indent=2))
    del encoder, embeddings
    if torch.cuda.is_available(): torch.cuda.empty_cache()


## 8. Findings and implementation proposals
The solution provides an inspectable workflow with predictions outside training groups and a separate path for partially coded records. The sample is small and simulated, and many classes have limited support. These results support a demonstration of the workflow rather than automatic coding acceptance in production.

Further evaluation needs a representative sample with respondent identifiers and a timestamp for each questionnaire version. Create an expert-adjudicated reference sample, specify the classification dictionary version and establish which fields are available at prediction time. Then compare this baseline with a Hebrew language model and candidate retrieval from an authoritative dictionary, using the same partition and metrics. These are research proposals rather than methods evaluated in this notebook.

A first deployment study could assist coders by displaying candidates and recording decisions and reasons, coding errors and handling time. Real-data evaluation and calibration would precede consideration of an automatic acceptance rule. Assess rare classes and missing fields separately, and monitor changes in job descriptions and the classification dictionary.

## 9. Sources and scope
Requirements come from the original brief and workbook sheets. Software documentation supports implementation details. Numerical findings come from the executed sample analysis.

* ILO, ISCO-08, occupation definition and hierarchy: https://isco.ilo.org/en/isco-08/
* CBS classification dictionaries: https://www.cbs.gov.il/he/Pages/Code-list.aspx
* CBS announcement of 30 April 2026 concerning a draft occupation classification: https://www.cbs.gov.il/he/mediarelease/DocLib/2026/132/20_26_132b.pdf
* scikit-learn, leakage prevention: https://scikit-learn.org/stable/common_pitfalls.html
* scikit-learn, GroupKFold: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html
* scikit-learn, text features: https://scikit-learn.org/stable/modules/feature_extraction.html
* scikit-learn, F1: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html
* scikit-learn, LinearSVC: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
* Google Colab, notebook execution: https://research.google.com/colaboratory/faq.html

Sources were reviewed on 30 October 2026. Metadata specifies a four-digit code, which defines this notebook's target. The announcement of a newer classification does not change the supplied labels. Each code has not been independently checked against an official dictionary. The data are not assumed to match international ISCO without Israeli adaptations.


## Figures from the saved results

These figures describe the saved execution. Rerun the calculations after changing the data or code. The rendering code is in the CBS section of Figures.ipynb.

![model_comparison](results/figures/model_comparison.png)

![split_sensitivity](results/figures/split_sensitivity.png)

![class_support](results/figures/class_support.png)

![missing_text](results/figures/missing_text.png)
